# Media Framing Batch API Preparation

This notebook prepares the final thesis batch input for the media-framing codebook.

It keeps the Katinka-style context-unit logic:
- one request per merged context window
- multiple media hits inside one window stay in the same row via `hit_text`
- `row_id` is preserved end-to-end
- Tagesschau is included, but direct Tagesschau self-references are removed

The notebook does not submit anything unless you explicitly set the final flag to `True`.


In [ ]:
from __future__ import annotations

import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd() / "2a_NER",
    Path.cwd(),
    Path.cwd().parent / "2a_NER",
    Path("/Users/MattisHaumann/Dev/Thesis/2a_NER"),
]
NOTEBOOK_DIR = next(
    (path for path in NOTEBOOK_DIR_CANDIDATES if (path / "media_framing_batch_utils.py").exists()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Could not locate 2a_NER/media_framing_batch_utils.py from the current working directory.")

PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from media_framing_batch_utils import (
    build_batch_requests_df,
    compile_master_pattern,
    create_batch_job,
    download_openai_file,
    estimate_batch_cost,
    extract_media_contexts,
    parse_batch_output_file,
    read_env_value,
    retrieve_batch_job,
    split_batch_requests_by_estimated_tokens,
    upload_batch_file,
    validate_batch_requests,
    write_batch_jsonl,
    write_manifest_csv,
)

DATA_PATH = NOTEBOOK_DIR / "df_combined.csv"
PROMPT_PATH = NOTEBOOK_DIR / "framing_codebook_prompt.txt"
OUTPUT_ROOT = NOTEBOOK_DIR / "outputs" / "batch_media_framing"
OUTPUT_DIR = OUTPUT_ROOT / "thesis_final" / "batch_workflow"
LEGACY_OUTPUT_DIR = OUTPUT_ROOT / "archive_old_tests" / "legacy_batches"
FULL_BATCH_JSONL_PATH = OUTPUT_DIR / "media_framing_thesis_batch.jsonl"
FULL_MANIFEST_PATH = OUTPUT_DIR / "media_framing_thesis_manifest.csv"
FULL_RESULTS_PATH = OUTPUT_DIR / "media_framing_thesis_results.csv"
FULL_ERRORS_PATH = OUTPUT_DIR / "media_framing_thesis_errors.csv"
BATCH_JOB_PATH = OUTPUT_DIR / "media_framing_thesis_batch_job.json"
EXISTING_MANIFEST_PATH = LEGACY_OUTPUT_DIR / "media_framing_full_manifest.csv"
BATCH_JOB_GLOB = "media_framing_thesis_batch_part*_job.json"
PART_JSONL_PATHS = []
PART_MANIFEST_PATHS = []
UPLOAD_BATCH_JSONL_PATH = FULL_BATCH_JSONL_PATH
UPLOAD_MANIFEST_PATH = FULL_MANIFEST_PATH

MODEL_NAME = "gpt-5-mini"
WINDOW = 1
SAFE_BATCH_INPUT_TOKEN_LIMIT = 4_500_000  # Tier-1-safe default for gpt-5-mini under a 5,000,000 enqueued-token limit.

FRAME_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "category": {
            "type": "string",
            "enum": [
                "POSITIONS-/PARTEILICHKEITS-BIAS",
                "VERZERRUNG/MANIPULATION",
                "DISINFORMATION/FALSCHDARSTELLUNG",
                "VERSAGEN/INKOMPETENZ",
                "NEUTRAL",
                "IRRELEVANT",
            ],
        },
        "evidence": {"type": "string"},
    },
    "required": ["category", "evidence"],
}

ANALYSIS_INSTRUCTIONS = (
    "Return valid JSON that matches the schema exactly. "
    "Do not add any keys beyond category and evidence."
)

LEGACY_RESULT_COLUMNS = [
    "hit_id",
    "row_id",
    "source",
    "Title",
    "hit_text",
    "context_idx",
    "count_hits",
    "count_unique_entities",
    "context_window",
    "model",
    "response_id",
    "category",
    "evidence",
    "raw_response_json",
]

for required_path in [DATA_PATH, PROMPT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Prompt path: {PROMPT_PATH}")
print(f"Batch JSONL path: {FULL_BATCH_JSONL_PATH}")
print(f"Manifest path: {FULL_MANIFEST_PATH}")


## 1. Load the Combined Data and the Codebook Prompt

In [ ]:
df = pd.read_csv(DATA_PATH)

if "row_id" not in df.columns:
    df = df.reset_index().rename(columns={"index": "row_id"})

required_columns = {"row_id", "source", "Title", "Text"}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"df_combined.csv is missing required columns: {missing_columns}")

search_columns = [column for column in ["row_id", "source", "Date", "Title", "Text"] if column in df.columns]
search_df = df[search_columns].copy()
search_df["source"] = search_df["source"].fillna("").astype(str)

CODEBOOK_PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()
if not CODEBOOK_PROMPT:
    raise ValueError(f"Prompt file is empty: {PROMPT_PATH}")

print(f"Articles loaded: {len(search_df):,}")
print(f"Unique sources: {search_df['source'].nunique():,}")
display(search_df.head(3))


## 2. Extract Context Windows and Run Thesis-Facing Checks

This is the key preparation step. It applies the mainstream-media filter, keeps Tagesschau in the data, and removes only direct Tagesschau self-references.


In [ ]:
MASTER_PATTERN = compile_master_pattern()

extraction = extract_media_contexts(search_df, pattern=MASTER_PATTERN, window=WINDOW)
media_context_df = extraction["media_context_df"].copy()
media_article_df = extraction["media_article_df"].copy()
kept_hits_df = extraction["kept_hits_df"].copy()
excluded_hits_df = extraction["excluded_hits_df"].copy()

required_context_columns = [
    "row_id",
    "source",
    "Title",
    "hit_text",
    "context_idx",
    "count_hits",
    "count_unique_entities",
    "context_window",
]
missing_context_columns = [column for column in required_context_columns if column not in media_context_df.columns]
if missing_context_columns:
    raise ValueError(f"media_context_df is missing required columns: {missing_context_columns}")
if media_context_df.empty:
    raise ValueError("media_context_df is empty after extraction.")
if media_context_df["row_id"].isna().any():
    raise AssertionError("row_id is missing in media_context_df.")
if media_context_df.duplicated(["row_id", "context_idx"]).any():
    raise AssertionError("row_id/context_idx pairs must be unique.")
if media_context_df["hit_text"].fillna("").eq("").any():
    raise AssertionError("Empty hit_text values found in media_context_df.")

self_term_pattern = r"(^| \| )(Tagesschau|Tagesschau24|ARD|Das Erste|das Erste)( \| |$)"
tagesschau_self_leaks_df = media_context_df.loc[
    (media_context_df["source"] == "Tagesschau")
    & (media_context_df["hit_text"].str.contains(self_term_pattern, regex=True, na=False)),
    ["row_id", "Title", "hit_text", "context_window"],
].copy()
if not tagesschau_self_leaks_df.empty:
    raise AssertionError("Tagesschau self-references still leaked into the final context rows.")

outlet_summary_df = (
    media_context_df.groupby("source", as_index=False)
    .agg(
        context_rows=("row_id", "size"),
        unique_articles=("row_id", "nunique"),
        multi_hit_windows=("count_unique_entities", lambda values: int((values > 1).sum())),
    )
    .sort_values(["context_rows", "unique_articles", "source"], ascending=[False, False, True])
    .reset_index(drop=True)
)

comparison_summary_df = pd.DataFrame()
changed_rows_df = pd.DataFrame()
if EXISTING_MANIFEST_PATH.exists():
    existing_manifest_df = pd.read_csv(EXISTING_MANIFEST_PATH)
    comparison_summary_df = pd.DataFrame(
        [
            {"metric": "existing_manifest_rows", "value": len(existing_manifest_df)},
            {"metric": "new_context_rows", "value": len(media_context_df)},
            {"metric": "row_delta", "value": len(media_context_df) - len(existing_manifest_df)},
            {
                "metric": "existing_tagesschau_rows",
                "value": int((existing_manifest_df["source"] == "Tagesschau").sum()),
            },
            {
                "metric": "new_tagesschau_rows",
                "value": int((media_context_df["source"] == "Tagesschau").sum()),
            },
        ]
    )

    changed_rows_df = existing_manifest_df[
        ["row_id", "context_idx", "source", "hit_text", "count_unique_entities"]
    ].merge(
        media_context_df[
            ["row_id", "context_idx", "source", "hit_text", "count_unique_entities"]
        ],
        on=["row_id", "context_idx", "source"],
        how="outer",
        suffixes=("_old", "_new"),
        indicator=True,
    )
    changed_rows_df = changed_rows_df.loc[
        (changed_rows_df["_merge"] != "both")
        | (changed_rows_df["hit_text_old"].fillna("") != changed_rows_df["hit_text_new"].fillna(""))
        | (
            changed_rows_df["count_unique_entities_old"].fillna(-1)
            != changed_rows_df["count_unique_entities_new"].fillna(-1)
        )
    ].reset_index(drop=True)

print(f"Context rows: {len(media_context_df):,}")
print(f"Unique filtered articles: {len(media_article_df):,}")
print(f"Excluded Tagesschau self-hits: {len(excluded_hits_df.loc[excluded_hits_df['exclusion_reason'] == 'tagesschau_self_reference']):,}")
display(outlet_summary_df)
if not comparison_summary_df.empty:
    display(comparison_summary_df)
if not changed_rows_df.empty:
    display(changed_rows_df.head(20))


## 3. Build the Batch Requests and Run a Parser Smoke Test

This check makes sure the batch output can later be converted back into the same result-row structure as the earlier synchronous GPT run.


In [ ]:
batch_requests_df = build_batch_requests_df(
    media_context_df,
    prompt_template=CODEBOOK_PROMPT,
    model_name=MODEL_NAME,
    analysis_instructions=ANALYSIS_INSTRUCTIONS,
    frame_schema=FRAME_SCHEMA,
)

validation_errors = validate_batch_requests(batch_requests_df)
if validation_errors:
    raise ValueError("Batch validation failed:\n" + "\n".join(validation_errors[:20]))

manifest_df = batch_requests_df[
    [
        "custom_id",
        "hit_id",
        "row_id",
        "source",
        "Title",
        "hit_text",
        "context_idx",
        "count_hits",
        "count_unique_entities",
        "context_window",
    ]
].copy()

if manifest_df["row_id"].isna().any():
    raise AssertionError("row_id is missing in the manifest.")
if manifest_df["hit_id"].duplicated().any():
    raise AssertionError("hit_id values are not unique.")
if manifest_df["custom_id"].duplicated().any():
    raise AssertionError("custom_id values are not unique.")

smoke_record = {
    "custom_id": manifest_df.iloc[0]["custom_id"],
    "response": {
        "status_code": 200,
        "body": {
            "id": "resp_smoke_test",
            "output_text": json.dumps({"category": "NEUTRAL", "evidence": ""}),
        },
    },
}

with tempfile.TemporaryDirectory() as tmpdir:
    smoke_output_path = Path(tmpdir) / "smoke_output.jsonl"
    smoke_output_path.write_text(json.dumps(smoke_record) + "\n", encoding="utf-8")
    parsed_results_df, parsed_errors_df = parse_batch_output_file(
        smoke_output_path,
        manifest_df,
        default_model_name=MODEL_NAME,
    )

if not parsed_errors_df.empty:
    raise AssertionError("The parser smoke test returned errors.")
if parsed_results_df.columns.tolist() != LEGACY_RESULT_COLUMNS:
    raise AssertionError(parsed_results_df.columns.tolist())

print(f"Batch requests ready: {len(batch_requests_df):,}")
display(manifest_df.head(5))
display(parsed_results_df.head(1))


## 4. Write the Final Files and Estimate Batch Cost

The pricing constants below are easy to update if OpenAI changes pricing later.

Note on split sizes:
The split parts can look very uneven in number of rows. That is expected because the notebook splits by estimated input tokens, not by request count. Context windows vary a lot in length, so `part01` is filled until it reaches the safer token threshold and `part02` contains the remaining rows.


In [ ]:
write_batch_jsonl(batch_requests_df, FULL_BATCH_JSONL_PATH)
write_manifest_csv(batch_requests_df, FULL_MANIFEST_PATH)

batch_parts = split_batch_requests_by_estimated_tokens(
    batch_requests_df,
    max_estimated_input_tokens=SAFE_BATCH_INPUT_TOKEN_LIMIT,
)
PART_JSONL_PATHS = []
PART_MANIFEST_PATHS = []
for idx, part_df in enumerate(batch_parts, start=1):
    part_jsonl_path = OUTPUT_DIR / f"media_framing_thesis_batch_part{idx:02d}.jsonl"
    part_manifest_path = OUTPUT_DIR / f"media_framing_thesis_manifest_part{idx:02d}.csv"
    write_batch_jsonl(part_df, part_jsonl_path)
    write_manifest_csv(part_df, part_manifest_path)
    PART_JSONL_PATHS.append(part_jsonl_path)
    PART_MANIFEST_PATHS.append(part_manifest_path)

BATCH_INPUT_PRICE_PER_MILLION = 0.25
BATCH_OUTPUT_PRICE_PER_MILLION = 2.00
OUTPUT_TOKEN_SCENARIOS = [40, 80]

cost_rows = []
for output_tokens_per_request in OUTPUT_TOKEN_SCENARIOS:
    estimate = estimate_batch_cost(
        batch_requests_df,
        output_tokens_per_request=output_tokens_per_request,
        batch_input_price_per_million=BATCH_INPUT_PRICE_PER_MILLION,
        batch_output_price_per_million=BATCH_OUTPUT_PRICE_PER_MILLION,
    )
    cost_rows.append(
        {
            "batch_label": "full",
            "assumed_output_tokens_per_request": output_tokens_per_request,
            "requests": estimate.n_requests,
            "estimated_input_tokens": estimate.estimated_input_tokens,
            "estimated_output_tokens": estimate.estimated_output_tokens,
            "estimated_total_cost_usd": round(estimate.estimated_total_cost_usd, 4),
        }
    )

    for part_idx, part_df in enumerate(batch_parts, start=1):
        part_estimate = estimate_batch_cost(
            part_df,
            output_tokens_per_request=output_tokens_per_request,
            batch_input_price_per_million=BATCH_INPUT_PRICE_PER_MILLION,
            batch_output_price_per_million=BATCH_OUTPUT_PRICE_PER_MILLION,
        )
        cost_rows.append(
            {
                "batch_label": f"part{part_idx:02d}",
                "assumed_output_tokens_per_request": output_tokens_per_request,
                "requests": part_estimate.n_requests,
                "estimated_input_tokens": part_estimate.estimated_input_tokens,
                "estimated_output_tokens": part_estimate.estimated_output_tokens,
                "estimated_total_cost_usd": round(part_estimate.estimated_total_cost_usd, 4),
            }
        )

cost_estimate_df = pd.DataFrame(cost_rows)

submitted_batch_job_paths = sorted(OUTPUT_DIR.glob(BATCH_JOB_GLOB))
submitted_batch_labels = set()
for job_path in submitted_batch_job_paths:
    batch_job = json.loads(job_path.read_text(encoding="utf-8"))
    metadata = batch_job.get("metadata") or {}
    submitted_batch_labels.add(
        metadata.get(
            "batch_label",
            job_path.stem.replace("media_framing_thesis_batch_", "").replace("_job", ""),
        )
    )

next_upload_index = None
for idx, part_jsonl_path in enumerate(PART_JSONL_PATHS):
    batch_label = part_jsonl_path.stem.replace("media_framing_thesis_batch_", "")
    if batch_label not in submitted_batch_labels:
        next_upload_index = idx
        break

UPLOAD_BATCH_JSONL_PATH = (
    PART_JSONL_PATHS[next_upload_index]
    if next_upload_index is not None
    else None
)
UPLOAD_MANIFEST_PATH = (
    PART_MANIFEST_PATHS[next_upload_index]
    if next_upload_index is not None
    else None
)

print(f"Batch JSONL written to: {FULL_BATCH_JSONL_PATH}")
print(f"Manifest written to: {FULL_MANIFEST_PATH}")
print(f"Safe split threshold (estimated input tokens): {SAFE_BATCH_INPUT_TOKEN_LIMIT:,}")
print(f"Split batch parts written: {len(PART_JSONL_PATHS):,}")
print(
    "Submitted parts already tracked: "
    + (", ".join(sorted(submitted_batch_labels)) if submitted_batch_labels else "none")
)
print(f"Default upload target: {UPLOAD_BATCH_JSONL_PATH}")
print("This notebook now defaults to a Tier-1-safe split for gpt-5-mini. Increase SAFE_BATCH_INPUT_TOKEN_LIMIT only if your OpenAI org has a higher batch queue limit.")
if UPLOAD_BATCH_JSONL_PATH is None:
    print("All parts already have job files. No next upload target is currently selected.")
display(cost_estimate_df)


## 5. Optional: Upload the Next Batch Part and Create the Batch Job

Leave this disabled unless you intentionally want to submit the next part. After a part has been submitted, set the flag back to `False` so status checks and parsing cannot accidentally create duplicate jobs.


In [ ]:
RUN_UPLOAD_AND_CREATE_BATCH = False

if RUN_UPLOAD_AND_CREATE_BATCH:
    api_key, api_key_source = read_env_value("OPENAI_API_KEY", project_root=PROJECT_ROOT)
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or export it in your shell.")
    if UPLOAD_BATCH_JSONL_PATH is None or UPLOAD_MANIFEST_PATH is None:
        raise RuntimeError("No next upload target is selected. Check existing job files and rerun the split/write cell.")

    upload_label = UPLOAD_BATCH_JSONL_PATH.stem.replace("media_framing_thesis_batch_", "")
    if upload_label == "batch":
        upload_label = "full"
    part_batch_job_path = OUTPUT_DIR / f"media_framing_thesis_batch_{upload_label}_job.json"
    if part_batch_job_path.exists():
        raise FileExistsError(
            f"A job file already exists for {upload_label}: {part_batch_job_path}. Refusing to resubmit this part."
        )

    upload_response = upload_batch_file(api_key, UPLOAD_BATCH_JSONL_PATH)
    batch_response = create_batch_job(
        api_key,
        input_file_id=upload_response["id"],
        metadata={
            "project": "thesis_media_framing",
            "manifest": UPLOAD_MANIFEST_PATH.name,
            "model": MODEL_NAME,
            "batch_label": upload_label,
        },
    )
    part_batch_job_path.write_text(json.dumps(batch_response, ensure_ascii=False, indent=2), encoding="utf-8")
    BATCH_JOB_PATH.write_text(json.dumps(batch_response, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"API key source: {api_key_source}")
    print(f"Uploaded JSONL path: {UPLOAD_BATCH_JSONL_PATH}")
    print(f"Uploaded input file id: {upload_response['id']}")
    print(f"Created batch id: {batch_response['id']}")
    print(f"Part-specific batch job metadata written to: {part_batch_job_path}")
    print(f"Latest batch job metadata also written to: {BATCH_JOB_PATH}")
else:
    print(
        "Upload disabled. Turn this on only when you intentionally want to submit the next part."
    )


## 6. Check Batch Status for Submitted Parts

Run this after submission to see which part is `in_progress`, `completed`, or `failed`. It reads all saved part-specific batch job JSON files in `thesis_final/batch_workflow/`.


In [ ]:
CHECK_BATCH_STATUS = False

part_batch_job_paths = sorted(OUTPUT_DIR.glob(BATCH_JOB_GLOB))

if CHECK_BATCH_STATUS:
    api_key, api_key_source = read_env_value("OPENAI_API_KEY", project_root=PROJECT_ROOT)
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or export it in your shell.")
    if not part_batch_job_paths:
        raise ValueError("No part-specific batch job JSON files found yet. Submit a part first.")

    status_rows = []
    for job_path in part_batch_job_paths:
        batch_job = json.loads(job_path.read_text(encoding="utf-8"))
        batch_status = retrieve_batch_job(api_key, batch_job["id"])
        status_output_path = OUTPUT_DIR / f"{job_path.stem}_status.json"
        status_output_path.write_text(json.dumps(batch_status, ensure_ascii=False, indent=2), encoding="utf-8")

        request_counts = batch_status.get("request_counts") or {}
        metadata = batch_status.get("metadata") or {}
        status_rows.append(
            {
                "batch_label": metadata.get("batch_label", job_path.stem.replace("media_framing_thesis_batch_", "").replace("_job", "")),
                "batch_id": batch_status.get("id", ""),
                "status": batch_status.get("status", ""),
                "created_at": batch_status.get("created_at"),
                "completed": request_counts.get("completed", 0),
                "failed": request_counts.get("failed", 0),
                "total": request_counts.get("total", 0),
                "output_file_id": batch_status.get("output_file_id", ""),
                "error_file_id": batch_status.get("error_file_id", ""),
                "job_json_path": str(job_path),
                "status_json_path": str(status_output_path),
            }
        )

    batch_status_df = pd.DataFrame(status_rows).sort_values("batch_label").reset_index(drop=True)
    print(f"API key source: {api_key_source}")
    display(batch_status_df)
else:
    print("Status check disabled. Set CHECK_BATCH_STATUS = True when you want to poll submitted parts.")


## 7. Optional: Download, Parse, and Combine Completed Batch Parts

Once one or more parts are completed, this cell downloads the available output files, parses them back into the legacy result-row shape, and writes combined result/error CSVs for analysis.


In [ ]:
DOWNLOAD_AND_PARSE_RESULTS = False
PARSE_COMPLETED_PARTS_ONLY = True

part_batch_job_paths = sorted(OUTPUT_DIR.glob(BATCH_JOB_GLOB))

if DOWNLOAD_AND_PARSE_RESULTS:
    api_key, api_key_source = read_env_value("OPENAI_API_KEY", project_root=PROJECT_ROOT)
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or export it in your shell.")
    if not part_batch_job_paths:
        raise ValueError("No part-specific batch job JSON files found yet. Submit at least one part first.")

    combined_results = []
    combined_errors = []
    part_summary_rows = []

    for job_path in part_batch_job_paths:
        batch_job = json.loads(job_path.read_text(encoding="utf-8"))
        batch_status = retrieve_batch_job(api_key, batch_job["id"])
        metadata = batch_status.get("metadata") or {}
        batch_label = metadata.get("batch_label", job_path.stem.replace("media_framing_thesis_batch_", "").replace("_job", ""))
        manifest_path = OUTPUT_DIR / f"media_framing_thesis_manifest_{batch_label}.csv"
        status = batch_status.get("status", "")

        if status != "completed":
            if PARSE_COMPLETED_PARTS_ONLY:
                part_summary_rows.append({
                    "batch_label": batch_label,
                    "status": status,
                    "results_rows": 0,
                    "error_rows": 0,
                    "manifest_path": str(manifest_path),
                    "output_jsonl_path": "",
                })
                continue
            raise ValueError(f"Batch {batch_label} is not completed yet: {status}")

        output_file_id = batch_status.get("output_file_id")
        if not output_file_id:
            raise ValueError(f"Batch {batch_label} is completed but has no output_file_id.")

        downloaded_output_path = download_openai_file(
            api_key,
            output_file_id,
            OUTPUT_DIR / f"media_framing_thesis_output_{batch_label}.jsonl",
        )
        manifest_df = pd.read_csv(manifest_path)
        results_df, errors_df = parse_batch_output_file(
            downloaded_output_path,
            manifest_df,
            default_model_name=MODEL_NAME,
        )
        results_df = results_df[LEGACY_RESULT_COLUMNS]

        combined_results.append(results_df)
        combined_errors.append(errors_df)

        error_file_id = batch_status.get("error_file_id")
        if error_file_id:
            download_openai_file(
                api_key,
                error_file_id,
                OUTPUT_DIR / f"media_framing_thesis_error_{batch_label}.jsonl",
            )

        part_summary_rows.append({
            "batch_label": batch_label,
            "status": status,
            "results_rows": len(results_df),
            "error_rows": len(errors_df),
            "manifest_path": str(manifest_path),
            "output_jsonl_path": str(downloaded_output_path),
        })

    combined_results_df = (
        pd.concat(combined_results, ignore_index=True)
        if combined_results
        else pd.DataFrame(columns=LEGACY_RESULT_COLUMNS)
    )
    combined_errors_df = (
        pd.concat(combined_errors, ignore_index=True, sort=False)
        if combined_errors
        else pd.DataFrame()
    )

    if not combined_results_df.empty:
        combined_results_df = combined_results_df.sort_values(
            ["row_id", "context_idx", "source", "hit_text"],
            ascending=[True, True, True, True],
        ).reset_index(drop=True)
        if combined_results_df["hit_id"].duplicated().any():
            raise AssertionError("Duplicate hit_id values found after combining batch parts.")

    combined_results_df.to_csv(FULL_RESULTS_PATH, index=False, encoding="utf-8")
    combined_errors_df.to_csv(FULL_ERRORS_PATH, index=False, encoding="utf-8")
    part_summary_df = pd.DataFrame(part_summary_rows).sort_values("batch_label").reset_index(drop=True)

    print(f"API key source: {api_key_source}")
    print(f"Combined results written to: {FULL_RESULTS_PATH}")
    print(f"Combined errors written to: {FULL_ERRORS_PATH}")
    display(part_summary_df)
    display(combined_results_df.head(5))
    display(combined_errors_df.head(5))
else:
    print(
        "Parsing disabled. Set DOWNLOAD_AND_PARSE_RESULTS = True after one or more submitted parts have completed."
    )
